# **Parte 1: Diagnóstico e Limpeza (Básico)**


**1** Inspecione as dimensões e o resumo dos tipos de dados do DataFrame df_vendas usando .shape e .info().



In [ ]:
df_vendas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   cliente_id  8 non-null      int64  
 1   valor       8 non-null      float64
 2   categoria   8 non-null      object 
 3   data_hora   8 non-null      object 
 4   status      8 non-null      object 
 5   email       8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [ ]:
linhas, colunas = df_vendas.shape
print(f'Dimensões: {linhas} linhas x {colunas} colunas')

Dimensões: 8 linhas x 6 colunas


**2** Identifique os valores nulos e preencha os registros ausentes da coluna valor utilizando a mediana da respectiva categoria ou a mediana global com .fillna().
negrito


In [ ]:
df_vendas['valor'] = df_vendas['valor'].fillna(df_vendas['valor'].median())
df_vendas['valor'].isna().sum()
df_vendas.describe()


,cliente_id,valor
count,8.000000,8.000000
mean,102.625000,930.143750
std,1.407886,1094.574218
min,101.000000,89.900000
25%,101.750000,384.875000
50%,102.500000,615.250000
75%,103.250000,885.375000
max,105.000000,3500.750000


**3** Filtre e exiba apenas as transações com status 'Concluído' e valor superior a R$ 500,00 usando .loc[] ou .query().


In [ ]:
df_filtrado = df_vendas.query("valor > 500 and status == 'Concluído'")
df_filtrado.shape

(4, 6)

## **Parte 2: Cruzamento e Transformação (Intermediário)**

 **4**. Realize uma junção relacional (left merge) entre df_vendas e df_clientes utilizando a chave cliente_id

In [ ]:
df_merged = pd.merge(df_vendas, df_clientes, on='cliente_id', how='left')
df_merged

,cliente_id,valor,categoria,data_hora,status,email,nome,cidade
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro
2,103,615.25,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com,Ana Oliveira,Belo Horizonte
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com,Carlos Lima,Curitiba
5,102,615.25,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br,Lucas Mendes,Salvador
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com,Ana Oliveira,Belo Horizonte


**5**. Crie uma nova coluna chamada media_categoria contendo o valor médio de vendas por categoria sem reduzir o número de linhas do DataFrame (utilize .groupby() com .transform())

In [ ]:
df_merged.groupby('cidade').agg({'valor': ['sum', 'mean']})
df_merged['media_cidade'] = df_merged.groupby('cidade')['valor'].transform('mean')
df_merged[['categoria', 'valor', 'media_cidade']].head(5)

,categoria,valor,media_cidade
0,Eletronicos,3500.75,2350.375
1,Livros,189.50,402.375
2,Roupas,615.25,697.875
3,Eletronicos,1200.00,2350.375
4,Automotivo,450.00,450.000


**6**. Identifique quantos clientes usam o provedor de e-mail @gmail.com através do acessor de texto .str.contains().


In [ ]:
gmail = df_merged[df_merged['email'].str.contains('@gmail.com')]
print('Emails do Gmail:', len(gmail))

Emails do Gmail: 3


#**Parte 3: Análise Temporal e Agregação (Intermediário)**

**7**. Converta a coluna data_hora para o tipo datetime nativo com pd.to_datetime() e extraia o mês por extenso ou o nome do dia da semana utilizando o acessor .dt.

In [ ]:
df_merged['data_hora'] = pd.to_datetime(df_merged['data_hora'])
df_merged['mes'] = df_merged['data_hora'].dt.month_name()
df_merged['dia_da_semana'] = df_merged['data_hora'].dt.day_name()
df_merged[['data_hora', 'mes', 'dia_da_semana']].head()

,data_hora,mes,dia_da_semana
0,2024-01-15 10:23:00,January,Monday
1,2024-01-18 14:05:00,January,Thursday
2,2024-02-05 09:12:00,February,Monday
3,2024-02-20 16:40:00,February,Tuesday
4,2024-03-02 11:00:00,March,Saturday


**8**Construa uma tabela dinâmica com pd.pivot_table() apresentando o total (sum) do valor de vendas agrupado por categoria nas linhas e por cidade nas colunas, preenchendo eventuais valores nulos com zero.


In [ ]:
tabela_dinamica = pd.pivot_table(df_merged,
                                   values='valor',
                                   index='categoria',
                                   columns='cidade',
                                   aggfunc='sum',
                                   fill_value=0)
display(tabela_dinamica)

cidade,Belo Horizonte,Curitiba,Rio de Janeiro,Salvador,Sao Paulo
categoria,,,,,
Automotivo,0.00,450.0,0.00,0.0,0.00
Eletronicos,0.00,0.0,0.00,0.0,4700.75
Livros,0.00,0.0,804.75,0.0,0.00
Roupas,1395.75,0.0,0.00,89.9,0.00
